## 00 · Original MACCO four-loss baseline

Train a fresh baseline from original OpenAI CLIP ViT-B/32 weights on 1,000 COCO pairs for five epochs, then establish frozen evaluation cases and baseline scores.

Notebook outline only: markdown cells describe the planned Colab implementation. No training code, data manifests, measurements, checkpoints, or scores are included. The experiment has not been run.

Reference: [MACCO source at `97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae`](https://github.com/hiker-lw/MACCO/tree/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae). The author code is under the [MACCO MIT license](https://github.com/hiker-lw/MACCO/blob/97fbae1416e30eba84cc0b8209d9cb6bdedfe6ae/LICENSE). When implementing, preserve original notices and keep the needed code visible in this notebook.


## S01 · Environment and assets

Record the GPU, CUDA availability, PyTorch and torchvision versions, and required packages (`ftfy`, `regex`, and the dependencies used by the selected author code). Stage chosen images under `/content/macco_data/`. Obtain the CLIP tokenizer vocabulary and original OpenAI ViT-B/32 weights as data assets. Keep implementation inside this notebook when code is added.


## S02 · Tokenizer

Use the pinned MACCO `src/open_clip_code/tokenizer.py` as the reference for the CLIP BPE helpers, `SimpleTokenizer`, and `tokenize`. Record an explicit path to `bpe_simple_vocab_16e6.txt.gz` and check positive and reviewed-negative captions use the same tokenizer.


## S03 · CLIP and MACCO model

Use ViT-B/32 with QuickGELU, the pinned CLIP and MACCO model definitions, and both image and text encoders. For every independent run, reload the same original OpenAI CLIP checkpoint, then initialize new MACCO predictors with the recorded common seed. Record the checkpoint identifier and hash. Train both encoders from the first optimizer step; use float32 trainable parameters with AMP.


## S04 · Preprocessing and datasets

Preserve the author's image transforms, crop-aware box-to-patch conversion, text masking, annotation fields, and joint image–text path. Select one positive row per source image or frame. Freeze row IDs before training. Shared COCO counts: 1,000 train from `train2014`, 500 project validation and 500 project test from disjoint `val2014` rows. Shared Talk2Car counts: 1,000 train, 500 official labeled validation, and 500 project test from whole scenes reserved from the official labeled training set. Check image, frame, and scene overlap.


## S05 · Original losses

Use the pinned author definitions, masking, normalization, targets, and reductions for MCA, MIR, MLM, and MIM. Baseline coefficients: MCA 1.0, MIR 0.1, MLM 20.0, MIM 250.0. Compare the four unweighted loss values on one fixed batch with the pinned author implementation during development.


## S06 · Training and checkpoints

Use the pinned single-GPU joint training path with a fresh optimizer per run, AMP, one update per batch, and `drop_last=False`. Train five full epochs over 1,000 positive pairs (5,000 pair presentations). Learning rates: `5e-7` for pretrained encoders and `1e-3` for new predictors; warmup: 50 updates. Choose one common batch size after the separate 8/16/32 speed check; candidate 16 gives 63 updates per epoch and 315 total. Compute the final update budget from the loader length. Save resolved settings, per-term losses, rows per epoch, update count, and checkpoint path.


## S07 · Shared evaluation

Freeze cases, candidate captions, preprocessing, scoring rules, and noise seed before final testing. Use the same 500-row Talk2Car validation and test pools for all five runs; COCO-trained runs also use the same COCO validation and test pools and separate selected Table 1 and Table 21 evaluations. Report image-to-text and text-to-image Recall@1, Recall@5, and Recall@10 over fixed candidate pools. Score clean and fixed-noise inputs on the same cases. Record benchmark, split, condition, seed, metric, unit, candidate pool, and `n_examples`. Compare variants with baseline only after matching these keys.


## 00.02 · Check shared implementation

Once implemented, run S01–S07 top to bottom in a fresh Colab runtime. Print the pinned source revision, GPU, and PyTorch version. Confirm tokenizer, model, losses, trainer, and evaluator are defined in this notebook.


## 00.03 · Common settings

Record `pretrained=openai`, five epochs, candidate batch size 16, AMP, common seed, original checkpoint identifier and hash, learning rates, 50-update warmup, paths, and original loss coefficients. Lock the final batch size after 00.06; both CLIP towers must be trainable from step one.


## 00.04 · Prepare COCO splits

Read the author TSV. Select exactly 1,000 `train2014` positives, including the positive IDs with reviewed negatives needed by 02. Make disjoint 500-row validation and 500-row project test splits from eligible `val2014` rows. Keep annotation columns and stage images. Save `coco_train.tsv`, `coco_val.csv`, and `coco_test.csv`; verify row counts and image IDs.


## 00.05 · Prepare traffic and evaluation splits

Join Talk2Car-Expr labels to Talk2Car metadata by `command_token` to obtain scene and frame IDs. Freeze 1,000 train, 500 official validation, and 500 scene-reserved project test rows. Save `traffic_train.csv`, `traffic_val.csv`, and `traffic_test.csv`; check scene and frame disjointness. Freeze COCO, traffic, Table 1, and Table 21 evaluation cases and clean/noisy settings in `eval_config.yaml`.


## 00.06 · Smoke test and batch speed

Before training, compare initial encoder state with the original checkpoint, check finite forward and backward losses, one optimizer update, and checkpoint reload at batch size 8. Time 8, 16, and 32 after five untimed steps and 20 timed steps per size, synchronizing CUDA. Record GPU, throughput, peak memory, and completion in `batch_speed_check.csv`; check that CML, noisy, and traffic batches fit. Lock the fastest stable common size with memory headroom.


## 00.07 · Smoke test evaluation

Run the shared evaluator on small fixed COCO and Talk2Car validation samples. Exercise the selected Table 1 and Table 21 definitions. Record split and case counts; clean and noisy paired evaluations must use the same cases.


## 00.08 · Train baseline

Train the four original losses at MCA 1.0, MIR 0.1, MLM 20.0, and MIM 250.0 in a unique `runs/baseline` output directory. Check all five epochs and 1,000 rows per epoch. Save resolved configuration, four-term loss log, update count, and checkpoint path.


## 00.09 · Evaluate baseline

Evaluate the trained encoder state with the shared functions on frozen project test pools. Report both retrieval directions at Recall@1, Recall@5, and Recall@10; include clean/noisy conditions and the selected Table 1 and Table 21 metrics separately. Save `baseline_metrics.csv` with case counts and evaluation keys.


## 00.10 · Save handoff

Save `shared_run_config.yaml`, `eval_config.yaml`, frozen manifests, reviewed negatives when available, baseline training log, and baseline scores outside the Colab runtime. Keep the checkpoint through its planned evaluations. These are planned outputs; this outline does not create them.
